# **Predikcija kategorije proizvoda na osnovu naslova**

## Analiza skupa podataka i treniranje modela

**Autor:** Senka Radivojšić

**Biblioteke:** pandas, scikit-learn, sklearn.model_selection, sklearn.linear_model, sklearn.feature_extraction.text, sklearn.tree, sklearn.naive_bayes, sklearn.svm, sklearn.metrics, matplotlib.pyplot, seaborn

## **01. Uvod**

Cilj ovog zadatka je da razvijemo model mašinskog učenja koji će automatski predlagati odgovarajuću kategoriju za svaki novi proizvod na osnovu njegovog naziva.

Koristimo dataset `products.csv` koji sadrži kolone:

- Product ID - jedinstveni identifikator,
- Product Title - naziv proizvoda,
- Merchant ID - prodavac,
- Category Label - ciljna kategorija,
- Product Code - interni kod,
- Number of Views - broj pregleda,
- Merchant Rating - ocena prodavca,
- Listing Date - datum postavljanja.

Python alati i biblioteke:

- `pandas` za rad sa podacima,
- `scikit-learn` za kreiranje, treniranje i evaluaciju modela,
- `matplotlib` ili `seaborn` za vizualizaciju matrica zabune i rezultata.

## **02. Učitavanje i analiza skupa podataka**

U prvom delu analize sprovodimo sledeće korake:

- uvozimo sve potrebne biblioteke
- učitavamo fajl `product.csv` sa github repozitorijuma u pandas DataFrame,
- prikazujemo ukupan broj redova i kolona,
- proveravamo broj nedostajućih vrednosti,
- prikazujemo nekoliko nasumično odabranih redova kako bismo stekli uvid u strukturu i sadržaj podataka,
- koristimo metodu `info()` kako bi dobili detaljnije informacije o dataframe-u,
- izlistavamo imena svih kolona.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import joblib
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Load dataset from GitHub
url = "https://raw.githubusercontent.com/senkaradivojsic/ml-product-reviews-project/main/data/products.csv"
df = pd.read_csv(url)

# Number of columns and rows
print("Dataset shape (rows, columns):", df.shape)

# Count missing values per column
print("\nMissing values per column:")
print(df.isna().sum())

# Preview a few example messages and information about the dataset
print("\nFive random rows:")
print(df.sample(5, random_state=42))
print("\nDataset info:")
df.info()

# List columns names
print("\nColumn names:")
print(df.columns)

## **03. Čišćenje i priprema podataka**

U narednim koracima vršimo čišćenje i pripremu podataka:

- standardizujemo imena kolona radi lakše dalje analize,
- štampamo jedinstvene vrednosti kolone `category_label`,
- standardizujemo vrednosti u koloni `category_label`,
- uklanjamo kolone koje nam nisu potrebne metodom `drop()`,
- koristimo metodu `dropna()` kako bi obrisali nedostajuće vrednosti.


In [ ]:
# Standardize columns names
df.columns = (df.columns.str.strip().str.lower().str.replace('_', ' ').str.replace(r'\s+', '_', regex=True).str.strip('_'))
print(df.columns)
print("-" * 40)

# Display value counts for the 'category_label' column
print(df['category_label'].value_counts(dropna=False))
print("-" * 40)

# Standardize the 'category_label' column
df['category_label'] = df['category_label'].str.strip().str.lower()

df['category_label'] = df['category_label'].replace({
    'fridges': 'fridge freezers',
    'freezers': 'fridge freezers',
    'fridge': 'fridge freezers',
    'cpu': 'cpus',
    'mobile phone': 'mobile phones'
})

print(df['category_label'].value_counts(dropna=False))
print("-" * 40)

# Deleting unnecessary columns
#df = df.drop(columns=['product_id', 'merchant_id', 'product_code', 'merchant_rating', 'listing_date', 'number_of_views'])
print(df.sample(5, random_state=42))
print("-" * 40)

# Deleting missing values
df = df.dropna(subset=['category_label', 'product_title'])
print(df.shape)
print(df.isna().sum())

## **04. Podela podataka**

U ovom delu analize koristimo modul `sklearn.model_selection` iz `Scikit-learn` biblioteke.
Najpre razdvajamo podatke na:

- X - ulazne podatke, odnosno naziv proizvoda;

- y - ciljnu promenljivu, odnosno kategorije kojima proizvodi pripadaju.

Nakon toga koristimo funkciju `train_test_split()` kako bismo podelili skup podataka u odnosu 80:20, odnosno 80% za treniranje i 20% za testiranje modela.

Prilikom podele koristimo sledeće parametre:

- `random_state=42` - omogućava ponovljivost podele podataka;

- `stratify=y` - čuva približno isti odnos kategorija u trening i test skupu.

Na kraju štampamo veličinu treniranog i testiranog data seta.

In [ ]:
# Separate features (X) and labels (y)
X = df["product_title"]
y = df["category_label"]

# Split the data: 80% for training, 20% for testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Display sizes of the resulting sets
print("Training set size:", len(X_train))
print("Test set size:", len(X_test))

# Display class distribution in the training and test sets
print("Training set class distribution:")
print(y_train.value_counts())
print("-" * 40)
print("Test set class distribution:")
print(y_test.value_counts())

## **05. Treniranje više modela i vektorizacija**

Kako bismo uporedili uspešnost različitih algoritama za klasifikaciju, koristićemo četiri modela:

- LogisticRegression,
- MultinomialNB,
- DecisionTreeClassifier,
- LinearSVC.

Prolazimo kroz petlju kroz sve klasifikacione modele iz rečnika models. Za svaki model kreiramo Pipeline koji prvo pretvara tekstualne podatke u TF-IDF numeričke vrednosti, a zatim trenira klasifikator nad tim podacima. Nakon treniranja, svaki Pipeline čuvamo u rečniku trained_models pod odgovarajućim imenom modela.

In [ ]:
# Train different classification models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Naive Bayes": MultinomialNB(),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Linear SVC": LinearSVC(max_iter=1000)
}

trained_models = {}
for name, classifier in models.items():
    pipeline = Pipeline([
        ('tfidf', TfidfVectorizer()),
        ('classifier', classifier)
    ])
    pipeline.fit(X_train, y_train)
    trained_models[name] = pipeline

## **06. Evaluacija performansi**

U ovom delu analize vršimo evaluaciju svakog istreniranog modela pomoću sledećih metrika i metoda:

- `accuracy_score` - meri ukupnu tačnost modela
- `classification_report` - prikazuje preciznost, odziv i F1-score za svaku kategoriju
- `confusion_matrix` - prikazuje broj tačno i pogrešno klasifikovanih poruka.

Na kraju, rezultate evaluacije vizualizujemo pomoću funkcije `seaborn.heatmap()`, kako bismo lakše uočili uspešnost klasifikacije svakog modela.

In [ ]:
# Evaluate all models
for name, model in trained_models.items():

    prediction = model.predict(X_test)

    print(f"\n{name}")
    print("Accuracy:", accuracy_score(y_test, prediction))
    print("\nClassification Report:\n",
          classification_report(y_test, prediction))

    print("Confusion Matrix:\n",
          confusion_matrix(y_test, prediction))

    sns.heatmap(confusion_matrix(y_test, prediction), annot=True, fmt='d', cmap='Blues')
    plt.title(f'Confusion Matrix for {name}')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()

## **07. Finalni model**

U poslednjem koraku ovog dela analize kreiramo pipeline sa svim potrebnim transformacijama i odabranim modelom, treniramo model na celom skupu i na kraju čuvamo model u .pkl formatu pomoću `joblib` modlula.

In [ ]:
# Train the final model on the entire dataset
final_model = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('classifier', LinearSVC(max_iter=1000))
])

final_model.fit(X, y)


# Save the trained model to a file
joblib.dump(final_model, "model/product_classifier_final_model.pkl")
print(" Model trained and saved as 'model/product_classifier_final_model.pkl'")

## **09. Zaključak**

Nakon evaluacije četiri modela, na osnovu metrika Accuracy, Precision, Recall i F1-score, došla sam do sledećih zaključaka:

- Najbolju tačnost (Accuracy) ostvario je Linear SVC model, sa rezultatom od 0.98;
- Proizvod najbolje prepoznaje Linear SVC, koji je ostvario prosečan Recall od 0.97. Ima najmanje propusta u poređenju sa ostalim modelima te nam daje najbolje rezultate.

Na osnovu dobijenih rezultata, Linear SVC se pokazao kao najbolji model za klasifikaciju proizvoda, jer je ostvario najbolje ukupne performanse, uključujući i najviši prosečan F1-score od 0.98.